# 실전 — 실제 데이터로 검증: 실제 웹 접근 로그로 의심 IP 찾기 (ch12)

ch12 §12.3(Combined Log Format 파싱)~§12.3.3(비정상 요청 패턴) 를 **실제 공개 웹 접근 로그**(nginx, 5만여 줄) 에 그대로 적용한다. 정규식 한 줄로 로그를 컬럼으로 분해하고, IP 별로 집계해 스캐너·봇으로 의심되는 IP 를 찾는다.

> **인터넷 필요**: 실행 시 실제 로그(약 7MB)를 내려받는다. (Zeek `conn.log` 는 PCAP 에서 Zeek 이 생성하므로, 여기서는 서버가 직접 남기는 웹 접근 로그를 쓴다.)

핵심 관찰: (1) **정규식 한 줄**로 raw 로그가 IP·시각·메서드·URI·상태·UA 컬럼이 된다 — 이후는 pandas `groupby` 문제다. (2) 실제 로그에는 404 가 압도적으로 많다 — 존재하지 않는 경로를 훑는 **스캐닝** 흔적이다. (3) IP 별 요청량 + 4xx 비율로 집계하면 **정상 사용자와 다른 신호를 내는 IP** 가 드러난다(§12.3.3).

## 데이터셋 이야기 — 어떻게 수집됐나 (실제 웹 접근 로그)

공개된 실제 **nginx 웹 서버 접근 로그(2015)** 로, Combined Log Format 5만여 줄을 담는다. 정상 다운로드 요청과 함께, 존재하지 않는 경로를 대량으로 훑는 **스캐닝 흔적(404 다발)** 이 그대로 섞여 있다.

**시나리오**: 서버가 매일 남기는 원시 접근 로그에서 정규식 파싱과 IP 집계만으로 스캐너·봇을 골라낼 수 있는가.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# !pip install pandas matplotlib

from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

import urllib.request, ssl, os, re
import pandas as pd, matplotlib.pyplot as plt

## 1. 실제 웹 로그 내려받기

In [ ]:
URL = ('https://raw.githubusercontent.com/elastic/examples/master/'
       'Common%20Data%20Formats/nginx_logs/nginx_logs')
def fetch(local, url):
    if not os.path.exists(local):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90, context=ssl.create_default_context()) as r:
            open(local, 'wb').write(r.read())
    return local

path = fetch('nginx_access.log', URL)
print('첫 줄:', open(path, encoding='utf-8', errors='replace').readline().strip()[:100])

## 2. Combined Log Format 파싱 (ch12 §12.3.2)

정규식 하나로 한 줄을 IP·시각·메서드·URI·상태·UA 로 분해한다.

In [ ]:
# ── 실습 과제 ──
# 목표: 2. Combined Log Format 파싱 (ch12 §12.3.2) — 정규식 하나로 한 줄을 IP·시각·메서드·URI·상태·UA 로 분해한다.
# 만들 결과: PAT, rows, df, df['status']
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
PAT = ...
    ...

rows = ...
        open(path, encoding = ...
df = ...
df['status'] = ...
...
```
</details>


## 3. IP 별 집계로 의심 IP 찾기 (ch12 §12.3.3)

요청량이 많고 4xx(대개 404) 비율이 높은 IP 는 존재하지 않는 경로를 훑는 스캐너·봇일 가능성이 크다.

In [ ]:
# ── 실습 과제 ──
# 목표: 3. IP 별 집계로 의심 IP 찾기 (ch12 §12.3.3) — 요청량이 많고 4xx(대개 404) 비율이 높은 IP 는 존재하지 않는 경로를 훑는 스캐너·봇일 가능성이 크다.
# 만들 결과: g, requests, err_4xx_rate, unique_uri
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
g = ...
    requests = ...
    err_4xx_rate = ...
    unique_uri = ...
...

...
print(g.sort_values('requests', ascending = ...
...
```
</details>


In [ ]:
# 요청량 상위 IP 의 4xx 비율 시각화
top = g.sort_values('requests', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index[::-1], top['err_4xx_rate'][::-1],
        color=['#e74c3c' if v > 0.5 else '#1a5f7a' for v in top['err_4xx_rate'][::-1]])
ax.set_xlabel('4xx 오류 비율'); ax.set_xlim(0, 1)
ax.set_title('요청량 상위 IP 의 4xx 비율 (붉은색=0.5 초과, 스캐너 의심)')
plt.tight_layout(); plt.show()

## 4. 정리

1. **정규식 한 줄이 raw 로그를 분석 가능한 표로 바꾼다** — 이후 모든 탐지는 pandas `groupby` 로 표현된다(ch12·ch13).
2. **실제 로그는 404 로 가득하다.** 존재하지 않는 경로를 대량으로 요청하는 IP 는 취약 경로를 훑는 스캐너의 전형이다.
3. **집계가 곧 탐지다.** 단일 요청은 정상처럼 보여도, IP 별 요청량·4xx 비율·고유 URI 수로 묶으면 정상 사용자와 다른 IP 가 드러난다 — 이것이 ch13 SIEM 상관 분석으로 이어진다.

> Zeek `conn.log`·방화벽 로그·intel.log 를 같은 방식으로 파싱해 `uid`·IP 로 결합하면(§12.4·ch13), 한 IP 가 여러 소스에서 남긴 신호를 교차 검증할 수 있다.